# Generación de encoders.pkl y subcircuitos.csv
Ejecutar **una sola vez** antes de levantar el backend.

Produce:
- `backend/model/encoders.pkl` — LabelEncoders para el modelo
- `backend/data/subcircuitos.csv` — zonas con centroides y freq_subcircuito

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.preprocessing import LabelEncoder

In [2]:
# ── RUTAS — ajusta según donde tengas tus CSV ──────────────────────────────────
RUTA_TRAIN = r"C:\Users\jhono\Downloads\Proyecto Titulación\CSV\mdi_train.csv"
RUTA_TEST  = r"C:\Users\jhono\Downloads\Proyecto Titulación\CSV\mdi_test.csv"

# Salida
OUT_ENCODERS     = os.path.join("backend", "model", "encoders.pkl")
OUT_SUBCIRCUITOS = os.path.join("backend", "data", "subcircuitos.csv")

In [3]:
# ── COLUMNAS CATEGÓRICAS (igual que en notebook 03) ────────────────────────────
COLS_CATEGORICAS = [
    "codigo_distrito",
    "codigo_circuito",
    "codigo_subcircuito",
    "codigo_iccs",
    "macro_lugar",
    "flag_coord",
]

# ── MAPA DE MACRO-LUGAR (igual que en notebook 03) ─────────────────────────────
MAPA_LUGAR = {
    "ÁREAS DE ACCESO PÚBLICO":          "ESPACIO_PUBLICO",
    "VÍA PÚBLICA":                      "ESPACIO_PUBLICO",
    "ESPACIO PÚBLICO":                  "ESPACIO_PUBLICO",
    "PARQUES":                          "ESPACIO_PUBLICO",
    "TROCHA":                           "ESPACIO_PUBLICO",
    "BOSQUES":                          "ESPACIO_PUBLICO",
    "RÍO":                              "ESPACIO_PUBLICO",
    "TERRENOS BALDIOS":                 "ESPACIO_PUBLICO",
    "CANCHAS DE USO MULTIPLE":          "ESPACIO_PUBLICO",
    "ESTADIOS":                         "ESPACIO_PUBLICO",
    "ÁREA PRIVADA":                     "VIVIENDA_PRIVADA",
    "VIVIENDA/ALOJAMIENTO":             "VIVIENDA_PRIVADA",
    "VIVIENDA PARTICULAR":              "VIVIENDA_PRIVADA",
    "ESPACIO PRIVADO":                  "VIVIENDA_PRIVADA",
    "CONJUNTO HABITACIONAL":            "VIVIENDA_PRIVADA",
    "CASA DE LA VICTIMA":               "VIVIENDA_PRIVADA",
    "CASA DEL VICTIMARIO":              "VIVIENDA_PRIVADA",
    "CASA DE PROTECCIÓN":               "VIVIENDA_PRIVADA",
    "ÁREAS DEDICADAS AL COMERCIO":      "COMERCIO_SERVICIOS",
    "CENTROS COMERCIALES":              "COMERCIO_SERVICIOS",
    "ALMACENES":                        "COMERCIO_SERVICIOS",
    "RESTAURANTES":                     "COMERCIO_SERVICIOS",
    "BARES / DISCOTECAS":               "COMERCIO_SERVICIOS",
    "BANCOS / FINANCIERAS":             "COMERCIO_SERVICIOS",
    "GASOLINERAS":                      "COMERCIO_SERVICIOS",
    "FARMACIAS":                        "COMERCIO_SERVICIOS",
    "UNIDADES DE TRANSPORTE":           "TRANSPORTE",
    "TERMINALES":                       "TRANSPORTE",
    "AEROPUERTOS":                      "TRANSPORTE",
    "PUERTOS":                          "TRANSPORTE",
    "VÍAS DE PRIMER ORDEN":             "TRANSPORTE",
    "PEAJES":                           "TRANSPORTE",
    "INSTITUCIONES EDUCATIVAS":         "INSTITUCIONAL",
    "CENTROS DE SALUD":                 "INSTITUCIONAL",
    "INSTITUCIONES PÚBLICAS":           "INSTITUCIONAL",
    "ENTIDADES FINANCIERAS DEL ESTADO": "INSTITUCIONAL",
    "UNIDADES DE POLICÍA COMUNITARIA":  "INSTITUCIONAL",
}

In [4]:
# ── CARGA DE DATOS ─────────────────────────────────────────────────────────────
train = pd.read_csv(RUTA_TRAIN, sep=";", encoding="utf-8-sig", low_memory=False)
test  = pd.read_csv(RUTA_TEST,  sep=";", encoding="utf-8-sig", low_memory=False)

# Unimos train + test para que los encoders vean TODOS los valores posibles
df = pd.concat([train, test], ignore_index=True)
print(f"Total registros: {len(df):,}")
print(f"Columnas: {list(df.columns)}")

Total registros: 76,860
Columnas: ['anio', 'mes', 'dia_semana', 'es_fin_de_semana', 'hora', 'codigo_distrito', 'codigo_circuito', 'codigo_subcircuito', 'freq_subcircuito', 'codigo_iccs', 'macro_lugar', 'flag_coord', 'gravedad']


In [5]:
# ── RECONSTRUIR macro_lugar si no existe en el CSV ─────────────────────────────
if "macro_lugar" not in df.columns:
    if "tipo_lugar" in df.columns:
        df["macro_lugar"] = df["tipo_lugar"].map(MAPA_LUGAR).fillna("OTRO")
        print("macro_lugar reconstruida desde tipo_lugar")
    else:
        df["macro_lugar"] = "OTRO"
        print("⚠️  tipo_lugar no encontrada — macro_lugar = OTRO")
else:
    print("macro_lugar ya existe en el CSV ✓")

# flag_coord
if "flag_coord" not in df.columns:
    df["flag_coord"] = "IMPUTADO"
    print("⚠️  flag_coord no encontrada — se asume IMPUTADO")
else:
    print("flag_coord ya existe en el CSV ✓")

macro_lugar ya existe en el CSV ✓
flag_coord ya existe en el CSV ✓


In [6]:
# ── ENTRENAR Y GUARDAR ENCODERS ────────────────────────────────────────────────
encoders = {}

for col in COLS_CATEGORICAS:
    if col not in df.columns:
        print(f"⚠️  '{col}' no encontrada — se omite")
        continue
    le = LabelEncoder()
    le.fit(df[col].astype(str))
    encoders[col] = le
    print(f"✓  {col}: {len(le.classes_)} valores únicos")

os.makedirs(os.path.dirname(OUT_ENCODERS), exist_ok=True)
joblib.dump(encoders, OUT_ENCODERS)
print(f"\n✅ encoders.pkl guardado en: {OUT_ENCODERS}")

✓  codigo_distrito: 10 valores únicos
✓  codigo_circuito: 57 valores únicos
✓  codigo_subcircuito: 240 valores únicos
✓  codigo_iccs: 180 valores únicos
✓  macro_lugar: 7 valores únicos
✓  flag_coord: 2 valores únicos

✅ encoders.pkl guardado en: backend\model\encoders.pkl


In [7]:
# ── GENERAR TABLA DE SUBCIRCUITOS ──────────────────────────────────────────────
# Frecuencia histórica
freq = df.groupby("codigo_subcircuito").size().rename("freq_subcircuito").reset_index()

# Detectar columnas de coordenadas
lat_col = next((c for c in df.columns if c.lower() in ["latitud", "lat", "latitude", "y"]), None)
lng_col = next((c for c in df.columns if c.lower() in ["longitud", "lng", "longitude", "x"]), None)

if lat_col and lng_col:
    # Centroides desde coordenadas REALES
    mask_real = df["flag_coord"].astype(str).str.upper() == "REAL"
    df_real   = df[mask_real] if mask_real.sum() > 0 else df
    centroides = (
        df_real
        .groupby("codigo_subcircuito")[[lat_col, lng_col]]
        .mean()
        .rename(columns={lat_col: "lat", lng_col: "lng"})
        .reset_index()
    )
    print(f"✓  Centroides calculados ({mask_real.sum():,} coords reales usadas)")
else:
    centroides = pd.DataFrame({
        "codigo_subcircuito": df["codigo_subcircuito"].unique(),
        "lat": np.nan,
        "lng": np.nan,
    })
    print("⚠️  Sin columnas de coordenadas — lat/lng vacíos. Completar manualmente.")

# Agregar circuito y distrito
cod_sup = (
    df.groupby("codigo_subcircuito")[["codigo_circuito", "codigo_distrito"]]
    .first()
    .reset_index()
)

subcircuitos = (
    freq
    .merge(centroides, on="codigo_subcircuito", how="left")
    .merge(cod_sup,    on="codigo_subcircuito", how="left")
    .sort_values("freq_subcircuito", ascending=False)
    .reset_index(drop=True)
)

os.makedirs(os.path.dirname(OUT_SUBCIRCUITOS), exist_ok=True)
subcircuitos.to_csv(OUT_SUBCIRCUITOS, index=False, encoding="utf-8-sig")

print(f"✅ subcircuitos.csv guardado en: {OUT_SUBCIRCUITOS}")
print(f"   Total subcircuitos: {len(subcircuitos)}")
subcircuitos[["codigo_subcircuito", "freq_subcircuito", "lat", "lng"]].head(5)

⚠️  Sin columnas de coordenadas — lat/lng vacíos. Completar manualmente.
✅ subcircuitos.csv guardado en: backend\data\subcircuitos.csv
   Total subcircuitos: 240


,codigo_subcircuito,freq_subcircuito,lat,lng
0,192,2567,NaN,NaN
1,161,1043,NaN,NaN
2,0,1031,NaN,NaN
3,99,1015,NaN,NaN
4,200,985,NaN,NaN


In [8]:
# ── VERIFICACIÓN FINAL ─────────────────────────────────────────────────────────
enc_cargado = joblib.load(OUT_ENCODERS)
print("Encoders guardados:", list(enc_cargado.keys()))
print("\n🎉 Todo listo. Pasos siguientes:")
print("   1. Copia modelo_final.pkl a backend/model/")
print("   2. cd backend && uvicorn main:app --reload")

Encoders guardados: ['codigo_distrito', 'codigo_circuito', 'codigo_subcircuito', 'codigo_iccs', 'macro_lugar', 'flag_coord']

🎉 Todo listo. Pasos siguientes:
   1. Copia modelo_final.pkl a backend/model/
   2. cd backend && uvicorn main:app --reload
